In [2]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_PATH = '/content/drive/MyDrive/Brain-Tumor-Classification'

Mounted at /content/drive


In [3]:
import os
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

BASE_PATH = f'{PROJECT_PATH}/data/raw'
CLASSES = ['glioma', 'meningioma', 'notumor', 'pituitary']
CLASS_TO_LABEL = {cls: idx for idx, cls in enumerate(CLASSES)}

print(CLASS_TO_LABEL)

{'glioma': 0, 'meningioma': 1, 'notumor': 2, 'pituitary': 3}


In [4]:
#Preprocessing function

def load_and_preprocess_data(split, target_size, color_mode='grayscale'):
    """
    Load images from disk and preprocess them.

    Args:
        split: 'Training' or 'Testing'
        target_size: tuple (width, height),
        color_mode: 'grayscale' or 'rgb'

    Returns:
        X: numpy array of images, normalized
        y: numpy array of integer labels
    """
    images = []
    labels = []

    split_path = os.path.join(BASE_PATH, split)

    for cls in CLASSES:
        class_path = os.path.join(split_path, cls)
        image_files = os.listdir(class_path)

        for img_file in image_files:
            img_path = os.path.join(class_path, img_file)

            img = Image.open(img_path)

            if color_mode == 'grayscale':
                img = img.convert('L')
            elif color_mode == 'rgb':
                img = img.convert('RGB')
            else:
                raise ValueError("color_mode must be 'grayscale' or 'rgb'")

            img = img.resize(target_size)
            img_array = np.array(img)

            images.append(img_array)
            labels.append(CLASS_TO_LABEL[cls])

    X = np.array(images, dtype='float32') / 255.0
    y = np.array(labels)

    if color_mode == 'grayscale':
        X = np.expand_dims(X, axis=-1)  # add channel dimension -> (N, H, W, 1)

    return X, y

In [5]:
X_train_scratch, y_train_scratch = load_and_preprocess_data(
    split='Training',
    target_size=(150, 150),
    color_mode='grayscale'
)

X_test_scratch, y_test_scratch = load_and_preprocess_data(
    split='Testing',
    target_size=(150, 150),
    color_mode='grayscale'
)

print("Scratch CNN data shapes:")
print(f"X_train: {X_train_scratch.shape}, y_train: {y_train_scratch.shape}")
print(f"X_test: {X_test_scratch.shape}, y_test: {y_test_scratch.shape}")

Scratch CNN data shapes:
X_train: (5600, 150, 150, 1), y_train: (5600,)
X_test: (1600, 150, 150, 1), y_test: (1600,)


In [6]:
X_train_transfer, y_train_transfer = load_and_preprocess_data(
    split='Training',
    target_size=(224, 224),
    color_mode='rgb'
)

X_test_transfer, y_test_transfer = load_and_preprocess_data(
    split='Testing',
    target_size=(224, 224),
    color_mode='rgb'
)

print("Transfer Learning data shapes:")
print(f"X_train: {X_train_transfer.shape}, y_train: {y_train_transfer.shape}")
print(f"X_test: {X_test_transfer.shape}, y_test: {y_test_transfer.shape}")

Transfer Learning data shapes:
X_train: (5600, 224, 224, 3), y_train: (5600,)
X_test: (1600, 224, 224, 3), y_test: (1600,)


In [7]:
X_train_scratch, X_val_scratch, y_train_scratch, y_val_scratch = train_test_split(
    X_train_scratch, y_train_scratch,
    test_size=0.15,
    random_state=42,
    stratify=y_train_scratch
)

X_train_transfer, X_val_transfer, y_train_transfer, y_val_transfer = train_test_split(
    X_train_transfer, y_train_transfer,
    test_size=0.15,
    random_state=42,
    stratify=y_train_transfer
)

print("After validation split:")
print(f"Scratch -> Train: {X_train_scratch.shape}, Val: {X_val_scratch.shape}")
print(f"Transfer -> Train: {X_train_transfer.shape}, Val: {X_val_transfer.shape}")

After validation split:
Scratch -> Train: (4760, 150, 150, 1), Val: (840, 150, 150, 1)
Transfer -> Train: (4760, 224, 224, 3), Val: (840, 224, 224, 3)


###One-hot encode labels

In [8]:
NUM_CLASSES = 4

y_train_scratch_ohe = to_categorical(y_train_scratch, num_classes=NUM_CLASSES)
y_val_scratch_ohe = to_categorical(y_val_scratch, num_classes=NUM_CLASSES)
y_test_scratch_ohe = to_categorical(y_test_scratch, num_classes=NUM_CLASSES)

y_train_transfer_ohe = to_categorical(y_train_transfer, num_classes=NUM_CLASSES)
y_val_transfer_ohe = to_categorical(y_val_transfer, num_classes=NUM_CLASSES)
y_test_transfer_ohe = to_categorical(y_test_transfer, num_classes=NUM_CLASSES)

print("Example before encoding:", y_train_scratch[0])
print("Example after encoding:", y_train_scratch_ohe[0])

Example before encoding: 0
Example after encoding: [1. 0. 0. 0.]


###Save processed data to Drive

In [9]:
processed_path = f'{PROJECT_PATH}/data/processed'
os.makedirs(processed_path, exist_ok=True)

# Scratch CNN data
np.save(f'{processed_path}/X_train_scratch.npy', X_train_scratch)
np.save(f'{processed_path}/X_val_scratch.npy', X_val_scratch)
np.save(f'{processed_path}/X_test_scratch.npy', X_test_scratch)
np.save(f'{processed_path}/y_train_scratch.npy', y_train_scratch_ohe)
np.save(f'{processed_path}/y_val_scratch.npy', y_val_scratch_ohe)
np.save(f'{processed_path}/y_test_scratch.npy', y_test_scratch_ohe)

# Transfer Learning data
np.save(f'{processed_path}/X_train_transfer.npy', X_train_transfer)
np.save(f'{processed_path}/X_val_transfer.npy', X_val_transfer)
np.save(f'{processed_path}/X_test_transfer.npy', X_test_transfer)
np.save(f'{processed_path}/y_train_transfer.npy', y_train_transfer_ohe)
np.save(f'{processed_path}/y_val_transfer.npy', y_val_transfer_ohe)
np.save(f'{processed_path}/y_test_transfer.npy', y_test_transfer_ohe)

print("All processed data saved successfully.")

All processed data saved successfully.
